# 环节 09 · 训练管线（配套 Notebook）

> 配套长文：[环节09-训练管线详解.md](./环节09-训练管线详解.md)
> 定位：把"三阶段要多少数据""LoRA 到底训了多少参数""显存账怎么算""Chinchilla 最优是什么"算成数。纯 Python 标准库，零依赖。

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 三阶段对照 | §1 | 数据量/成本差几个数量级 |
| §2 LoRA 参数手算 | §4.3.3 | 7B 只训 1678 万 ≈ 0.24% |
| §3 低秩分解 | §4.3.2 | ΔW = B·A 为什么不物化全矩阵；B 零初始化的意义 |
| §4 显存账三件套 | §4.3.1 / §4.4.1 | FFT 98GB / LoRA 差在哪 / QLoRA 再省什么 |
| §5 什么时候微调 | §4.1 / §4.5 | 知识→RAG，行为→微调 |
| §6 Scaling Law | §2.2 | C ≈ 6ND 与 Chinchilla 的 20 tokens/参数 |
| §7 分布式与显存 | §5 | TP/PP/DP/ZeRO 各切什么 |


## 1. 三阶段：同样叫"训练"，量级差几百倍（长文 §1）

| 阶段 | 学什么 | 数据 | 产出 |
|---|---|---|---|
| 预训练 | 世界知识 + 语言 | 万亿 token | 基座（只会续写） |
| SFT | 指令服从与格式 | 几万条标注 | Chat 模型（会听话） |
| 对齐 | 偏好 / 价值观 / 推理风格 | 偏好对 + 规则信号 | 产品模型（R1 类） |


In [ ]:
stages = [
    ("预训练", "万亿 token", "千卡级 · 月级", "基座（只会续写）"),
    ("SFT", "几万条标注问答", "百卡级 · 天级", "Chat 模型（会听话）"),
    ("对齐（RLHF/DPO/GRPO）", "偏好对 / 规则信号", "百卡级 · 天级", "产品模型（R1 类）"),
]
print(f"{'阶段':<24} {'数据量':<16} {'算力规模':<16} 产出")
print("-" * 86)
for name, data, cost, out in stages:
    print(f"{name:<24} {data:<16} {cost:<16} {out}")

print("\n量级感受（预训练 vs SFT）：")
pretrain_tokens = 15e12          # 15 T tokens 量级
sft_samples = 5e4
print(f"  预训练：{pretrain_tokens:.0e} token")
print(f"  SFT   ：{sft_samples:.0e} 条样本（每条按 500 token 算 ≈ {sft_samples*500:.0e} token）")
print(f"  → 差了 {pretrain_tokens/(sft_samples*500):,.0f} 倍")
print("\n结论（长文 §2.3）：对 99% 的团队，不要自己预训练；预算花在数据、微调和 Agent 工程上。")


## 2. LoRA 到底训了多少参数（长文 §4.3.3）

`ΔW = B·A`，其中 `B ∈ R^{d×r}`、`A ∈ R^{r×k}`，`r ≪ min(d,k)`。注入到 attention 的 q/k/v/o 四个投影上：

```
每个投影：B(d×r) + A(r×d) = 2·d·r 个参数
```


In [ ]:
d, L, r = 4096, 32, 16
per_proj = 2 * d * r
per_layer = per_proj * 4            # q/k/v/o
total = per_layer * L

print(f"设定：hidden d={d}、{L} 层、注入 q/k/v/o、秩 r={r}\n")
print(f"  每个投影 B({d}×{r}) + A({r}×{d}) = {per_proj:>10,} 参数")
print(f"  每层 4 个投影              = {per_layer:>10,}")
print(f"  {L} 层合计                    = {total:>10,} ≈ {total/1e4:.0f} 万")
print(f"\n  占 7B 的比例 = {total/7e9:.4%}   （长文 §4.3.3：≈0.24%）")
print(f"  r=8 时       = {total/2/1e4:.0f} 万，占比 {total/2/7e9:.4%}"
      f"（长文：~840 万 / 0.12%）")

print(f"\n对照全参：7,000,000,000 个参数 → LoRA 是它的 1/{7e9/total:.0f}")
print("\n公式速记（长文面试题 11）：每模块 2·d·r × 注入模块数。")


## 3. 低秩分解凭什么省（长文 §4.3.2）

关键不是"参数少"，而是**整张 4096×4096 的 ΔW 从不物化**：计算时先 `A·x` 压到 r 维，再 `B·(A·x)` 展开。

另外两个设计细节：**B 零初始化**（起点就是原模型）、**α/r 缩放**（不同秩之间超参可迁移）。


In [ ]:
import random

random.seed(0)
r_small = 4
B = [[0.0] * r_small for _ in range(4)]                     # B 零初始化
A = [[random.gauss(0, 0.1) for _ in range(4)] for _ in range(r_small)]
x = [random.gauss(0, 1) for _ in range(4)]


def mv(M, v):
    return [sum(p * q for p, q in zip(row, v)) for row in M]


delta_w_x = mv(B, mv(A, x))
print(f"初始时 ΔW·x = B·(A·x) = {[round(v, 10) for v in delta_w_x]}")
print("→ 恒为 0：训练起点 = 原模型（不训就是原样），开关 LoRA 无痕（长文 §4.3.4）")

d_full, k_full = 4096, 4096
print(f"\n参数账（d=k={d_full}）：")
print(f"  全矩阵 ΔW        = {d_full}×{k_full} = {d_full*k_full/1e6:>8.1f} M")
print(f"  低秩 B·A (r=16)  = 2×{d_full}×16   = {2*d_full*16/1e6:>8.3f} M"
      f"   → 省 {d_full*k_full/(2*d_full*16):.0f} 倍")

print("\nα/r 缩放：")
for alpha, rr in [(16, 16), (32, 16), (16, 8)]:
    print(f"  alpha={alpha:<3} r={rr:<3} → scale = alpha/r = {alpha/rr:.4f}")
print("  → alpha 固定时 scale 只随 r 变，所以不同秩之间超参可以迁移（长文 §4.3.4）")


## 4. 显存账：LoRA 省的是哪一块（长文 §4.3.1）

**面试高频误区**：LoRA 不省激活（activation）。它省的只是"可训练参数的梯度 + 优化器状态"。


In [ ]:
P = 7e9          # 7B 参数
FFT_BYTES = 2 + 4 + 4 + 4      # BF16 权重 + fp32 主权重 + Adam 一阶/二阶矩
print("FFT（全参微调）的显存：")
print(f"  每参数 ≈ {FFT_BYTES} B（BF16 权重 2 + fp32 主权重 4 + Adam m 4 + v 4）")
print(f"  → 7B × {FFT_BYTES} B = {P*FFT_BYTES/1e9:>6.1f} GB（长文 ≈100GB+）")
print(f"  + 梯度（BF16 2 B/参数）→ {P*(FFT_BYTES+2)/1e9:>6.1f} GB")
print(f"  + 激活（另算，与 batch×序列长度 成正比）")

lora_trainable = 16777216
print(f"\nLoRA（r=16，q/k/v/o）的显存：")
print(f"  可训练参数 {lora_trainable/1e4:.0f} 万 → 梯度+优化器态 ≈ "
      f"{lora_trainable*(2+FFT_BYTES-2)/1e6:.0f} MB")
print(f"  冻结基座 BF16 = {P*2/1e9:.0f} GB（只参与前向/反传，**不存梯度、不更新**）")
print(f"  激活：**一样大** —— LoRA 不省激活，靠 activation checkpointing / 小 batch 压")

print(f"\n→ 落地门槛（长文 §4.3.1）：FFT 8×80G 起步；LoRA 单卡 24~48GB 可训 7B。")
print(f"\nQLoRA 再省什么（长文 §4.4.1）：把**冻结基座**从 BF16 压到 NF4 4bit：")
print(f"  基座 {P*2/1e9:.0f} GB → {P*0.5/1e9:.1f} GB（+ 双重量化再省 ~0.37 bit/参数）")
print("  + 分页优化器（显存不够时换页到 CPU）→ 单卡可训 7B~70B")


## 5. 什么时候该微调（长文 §4.1）

**知识 → RAG，行为 / 格式 / 风格 → 微调，临时指令 → Prompt。**


In [ ]:
decision = [
    ("事实知识（会变/长尾）", "RAG", "微调记不住，更新要重训"),
    ("私有长尾知识（内部规范）", "RAG 为主", "强灌知识 = 遗忘 + 幻觉"),
    ("固定输出格式（JSON/结构）", "微调 或 结构化输出", "格式硬且高频 → 微调"),
    ("风格/语气/领域术语", "微调", "行为层的改变微调最有效"),
    ("推理链格式/工具调用习惯", "微调", "让模型按你的套路走"),
    ("临时、一次性的要求", "Prompt", "不值得动权重"),
]
print(f"{'需求类型':<28} {'首选手段':<18} 说明")
print("-" * 80)
for need, way, note in decision:
    print(f"{need:<28} {way:<18} {note}")

print("\n微调工程的三条红线（长文 §4.5 / §6）：")
print("  ① epoch 1~3：轮次过多 = 灾难性遗忘，通用能力崩")
print("  ② 数据里混 10~20% 通用数据，保住通用能力")
print("  ③ 评估必做：任务分涨 + 通用分不掉才算成功（回归集见横切主题《模型评测与选型》）")


## 6. Scaling Law 与 Chinchilla（长文 §2.2）

```
训练算力  C ≈ 6·N·D          （N 参数、D token；前向 2ND + 反向 4ND）
Chinchilla 最优：D ≈ 20·N    →  C ≈ 120·N²  →  N* = √(C/120)
```


In [ ]:
print(f"{'算力 C (FLOPs)':>16} {'N* 参数':>12} {'D* token':>14} {'6ND 校验':>14} {'每参数 token':>14}")
print("-" * 74)
for C in (1e22, 1e23, 1e24, 1e25):
    N = (C / 120) ** 0.5
    D = 20 * N
    print(f"{C:>16.0e} {N/1e9:>10.1f} B {D/1e12:>12.2f} T {6*N*D:>14.2e} {D/N:>14.1f}")

print("\n→ 参数量和 token 数要**同步放大**（D/N = 20 是固定配比）；")
print("  只加参数不加数据 = 浪费算力（长文 §2.2）。")
print("\n案例对照：")
for name, N, D in [("GPT-3", 175e9, 300e9), ("Chinchilla", 70e9, 1.4e12),
                   ("LLaMA-2 7B", 7e9, 2e12), ("LLaMA-3 8B", 8e9, 15e12)]:
    print(f"  {name:<12} N={N/1e9:>5.0f}B  D={D/1e12:>5.2f}T  D/N = {D/N:>7.1f}"
          f"  C ≈ {6*N*D:.2e} FLOPs")
print("  → GPT-3 是典型的“参数大、数据少”（D/N ≈ 1.7，远低于 20），Llama 系普遍过训")
print("    （D/N 上百到上千）：推理成本摊薄，是工程上更划算的选择。")


## 7. 分布式训练：四种"切法"与显存（长文 §5）

**TP 切模型、PP 切层、DP 切数据、ZeRO 切状态。**


In [ ]:
plans = [
    ("DP / FSDP", "数据分片，每卡一份完整模型", "每步全量梯度 all-reduce", "默认起点"),
    ("TP", "单层参数切到多卡", "每层都通信", "单机多卡大模型"),
    ("PP", "按层分组，卡间接力", "层边界一次", "多机超大模型（有气泡）"),
    ("ZeRO-1/2/3", "优化器状态 → +梯度 → +参数 分片", "逐步增多", "显存不够时的救星"),
    ("EP", "MoE 专家分布多卡", "路由通信", "MoE 模型"),
]
print(f"{'方案':<12} {'切什么':<28} {'通信':<20} 适用")
print("-" * 84)
for name, what, comm, use in plans:
    print(f"{name:<12} {what:<28} {comm:<20} {use}")

print("\nZeRO 三阶段省的是哪块显存（长文 §5 表 + §4.3.1）：")
P = 7e9
for stage, desc, divisor in [("ZeRO-1", "切优化器状态", 4 + 4),
                             ("ZeRO-2", "+ 切梯度", 4 + 4 + 2),
                             ("ZeRO-3", "+ 切参数", 4 + 4 + 2 + 2)]:
    total_b = 14
    other = total_b - divisor
    print(f"  {stage}: {desc:<12} 每卡保留 = {other} B/参数（其余按卡数平分）"
          f" → 单卡 {P*other/1e9:>5.1f} GB 起步")

print("\n显存优化三板斧（长文 §5）：")
print("  ① 重计算（activation checkpointing）：反向时重算激活，省显存换算力")
print("  ② 混合精度（BF16/FP16）：训练标配，BF16 更稳")
print("  ③ ZeRO / 量化：优化器状态分片、4bit 基座")


## 8. 自测（长文 §7）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| 预训练和微调差多少？ | §1：数据量差几百倍 |
| LoRA 训了多少参数？ | §2：1678 万 ≈ 7B 的 0.24% |
| LoRA 为什么省显存？省的是哪块？ | §4：FFT 98GB vs LoRA 只训增量的梯度/优化器态；**激活不省** |
| QLoRA 又省了什么？ | §4：把冻结基座从 14GB 压到 3.5GB + 分页优化器 |
| 什么时候用 RAG、什么时候微调？ | §5：知识 → RAG，行为/格式/风格 → 微调 |
| Chinchilla 最优是什么？ | §6：D ≈ 20N，参数量与数据同步放大 |
| 分布式四种切法？ | §7：TP 切模型、PP 切层、DP 切数据、ZeRO 切状态 |

**上一站** [环节 08 · 输出头与训练目标](./环节08-输出头与训练目标详解.md)   **下一站** [环节 10 · 推理解码与 KV Cache](./环节10-推理解码与KV缓存详解.md)
